# Manifestation: Genesis and Evaporation

**"The threshold between potential and actual."**

This notebook explores the fundamental process of **manifestation** - how the continuous flux field gives rise to discrete particle states.

---

## The Core Mechanism

| Process | Transition | Condition |
|---------|------------|------------|
| **Genesis** | 0 → ±1 | |J| > KB (flux exceeds threshold) |
| **Evaporation** | ±1 → 0 | |J| < KB (flux below threshold) |

The polarity (+1 or -1) is determined by the **divergence** of flux: ∇·J

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label

repo_root = os.path.abspath("../../")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from ternary_matrix.model.grid import Universe
from ternary_matrix.physics import master_equation, waves, forces
from ternary_matrix.config import CONSTANTS

print("Modules loaded.")

## 1. The Manifestation Threshold (KB)

KB is the **energy threshold** for manifestation. In FTD:

$$K_B = m_e c^2$$

where $m_e$ is derived as:

$$m_e = m_P \sqrt{2\pi} \cdot \frac{16}{3} \cdot \alpha^{11}$$

Let's visualize the probability function:

In [ ]:
# Manifestation probability function
def genesis_probability(density, KB):
    """P(manifest) = 1 - exp(-(ρ - KB)/KB) for ρ > KB"""
    p = np.where(
        density > KB,
        1.0 - np.exp(-(density - KB) / KB),
        0.0
    )
    return np.clip(p, 0, 1)

# Visualize
KB = 1.0
densities = np.linspace(0, 5, 200)
probabilities = genesis_probability(densities, KB)

plt.figure(figsize=(10, 5))
plt.plot(densities, probabilities, 'b-', linewidth=2)
plt.axvline(KB, color='r', linestyle='--', label=f'KB = {KB}')
plt.fill_between(densities, probabilities, alpha=0.3)

plt.xlabel('Flux Density |J|', fontsize=12)
plt.ylabel('P(manifest)', fontsize=12)
plt.title('Genesis Probability: P = 1 - exp(-(ρ - KB)/KB)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 5)
plt.ylim(0, 1.1)

# Add annotations
plt.annotate('No manifestation\n(ρ < KB)', xy=(0.3, 0.05), fontsize=10, ha='center')
plt.annotate('Probabilistic\nmanifestaton', xy=(1.5, 0.5), fontsize=10, ha='center')
plt.annotate('Near-certain\nmanifestaton', xy=(4, 0.9), fontsize=10, ha='center')

plt.show()

## 2. Genesis Experiment: Matter from Flux

Let's inject high flux and watch matter appear.

In [ ]:
# Configure for observable genesis
CONSTANTS.C = 0.5
CONSTANTS.KB = 1.5
CONSTANTS.DECAY_RATE = 0.001
CONSTANTS.DAMPING = 0.02

universe = Universe(size=32)
center = universe.size // 2

# Create a high-flux region (supercritical)
for dx in range(-4, 5):
    for dy in range(-4, 5):
        for dz in range(-4, 5):
            r = np.sqrt(dx**2 + dy**2 + dz**2)
            if r < 5:
                # Radial outward flux (positive divergence at center)
                if r > 0:
                    direction = np.array([dx, dy, dz]) / r
                else:
                    direction = np.array([0, 0, 0])
                magnitude = 4.0 * np.exp(-r**2 / 8)
                universe.flux[center+dx, center+dy, center+dz, :] = magnitude * direction

# Calculate initial fields
forces.calculate_density(universe)

print(f"Initial state:")
print(f"  Max density: {universe.density.max():.2f}")
print(f"  Threshold KB: {CONSTANTS.KB}")
print(f"  Supercritical region: {np.sum(universe.density > CONSTANTS.KB)} voxels")

In [ ]:
# Run simulation and track populations
history = {'t': [], 'positive': [], 'negative': [], 'total': [], 'max_density': []}

for t in range(50):
    n_pos = np.count_nonzero(universe.states == 1)
    n_neg = np.count_nonzero(universe.states == -1)
    
    history['t'].append(t)
    history['positive'].append(n_pos)
    history['negative'].append(n_neg)
    history['total'].append(n_pos + n_neg)
    history['max_density'].append(universe.density.max())
    
    master_equation.tick(universe)

print(f"\nFinal state:")
print(f"  Positive particles: {history['positive'][-1]}")
print(f"  Negative particles: {history['negative'][-1]}")

In [ ]:
# Visualize genesis dynamics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Population evolution
ax = axes[0]
ax.plot(history['t'], history['positive'], 'r-', linewidth=2, label='Positive (+1)')
ax.plot(history['t'], history['negative'], 'b-', linewidth=2, label='Negative (-1)')
ax.plot(history['t'], history['total'], 'k--', linewidth=1, label='Total')
ax.set_xlabel('Time (ticks)', fontsize=12)
ax.set_ylabel('Number of Particles', fontsize=12)
ax.set_title('Genesis Events: Matter Emerging from Flux', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Density evolution
ax = axes[1]
ax.plot(history['t'], history['max_density'], 'g-', linewidth=2)
ax.axhline(CONSTANTS.KB, color='r', linestyle='--', label=f'KB = {CONSTANTS.KB}')
ax.set_xlabel('Time (ticks)', fontsize=12)
ax.set_ylabel('Max Flux Density', fontsize=12)
ax.set_title('Flux Density Decay', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Genesis: Flux → Matter Transition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Polarity Selection: The Role of Divergence

The sign of manifested particles is determined by **flux divergence**:

$$\nabla \cdot J > 0 \rightarrow +1 \text{ (matter)}$$
$$\nabla \cdot J < 0 \rightarrow -1 \text{ (antimatter)}$$

Let's visualize divergence:

In [ ]:
# Create universe with controlled divergence patterns
universe = Universe(size=48)
center = universe.size // 2

# Left side: positive divergence (outward flux) -> +1 particles
# Right side: negative divergence (inward flux) -> -1 particles

left_center = center - 10
right_center = center + 10

for dx in range(-5, 6):
    for dy in range(-5, 6):
        r = np.sqrt(dx**2 + dy**2) + 0.01
        if r < 6:
            # Left: radial outward (∇·J > 0)
            magnitude = 3.0 * np.exp(-r**2 / 10)
            direction_out = np.array([dx/r, dy/r, 0])
            universe.flux[left_center+dx, center+dy, center, :] = magnitude * direction_out
            
            # Right: radial inward (∇·J < 0)
            direction_in = np.array([-dx/r, -dy/r, 0])
            universe.flux[right_center+dx, center+dy, center, :] = magnitude * direction_in

# Calculate divergence
forces.calculate_density(universe)
divergence = forces.calculate_divergence(universe)

In [ ]:
# Visualize flux and divergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z = center

# Flux magnitude
flux_mag = np.linalg.norm(universe.flux[:, :, z, :], axis=-1)
im0 = axes[0].imshow(flux_mag.T, origin='lower', cmap='inferno')
axes[0].set_title('Flux Magnitude |J|', fontsize=12)
plt.colorbar(im0, ax=axes[0])

# Divergence
div_slice = divergence[:, :, z]
vmax = abs(div_slice).max()
im1 = axes[1].imshow(div_slice.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Divergence ∇·J (Red=+, Blue=-)', fontsize=12)
plt.colorbar(im1, ax=axes[1])

# Vector field
step = 2
X, Y = np.meshgrid(np.arange(0, universe.size, step), np.arange(0, universe.size, step))
Jx = universe.flux[::step, ::step, z, 0]
Jy = universe.flux[::step, ::step, z, 1]
axes[2].quiver(X, Y, Jx.T, Jy.T, np.sqrt(Jx**2 + Jy**2).T, cmap='viridis')
axes[2].set_title('Flux Vectors', fontsize=12)
axes[2].set_xlim(0, universe.size)
axes[2].set_ylim(0, universe.size)

# Add labels
axes[0].text(left_center, 5, '∇·J > 0', ha='center', color='white', fontsize=10)
axes[0].text(right_center, 5, '∇·J < 0', ha='center', color='white', fontsize=10)

plt.suptitle('Divergence Determines Polarity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Run simulation and observe polarity distribution
CONSTANTS.KB = 1.0

for t in range(20):
    master_equation.tick(universe)

# Count particles by location
left_mask = np.zeros(universe.shape, dtype=bool)
left_mask[:center, :, :] = True
right_mask = ~left_mask

left_positive = np.count_nonzero((universe.states == 1) & left_mask)
left_negative = np.count_nonzero((universe.states == -1) & left_mask)
right_positive = np.count_nonzero((universe.states == 1) & right_mask)
right_negative = np.count_nonzero((universe.states == -1) & right_mask)

print("Polarity Distribution:")
print(f"  Left side (∇·J > 0):  +1: {left_positive}, -1: {left_negative}")
print(f"  Right side (∇·J < 0): +1: {right_positive}, -1: {right_negative}")

In [ ]:
# Visualize final particle positions
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Get positions of manifested particles
pos_coords = np.argwhere(universe.states == 1)
neg_coords = np.argwhere(universe.states == -1)

if len(pos_coords) > 0:
    ax.scatter(pos_coords[:,0], pos_coords[:,1], pos_coords[:,2], 
               c='red', s=50, alpha=0.7, label=f'+1 Matter ({len(pos_coords)})')
if len(neg_coords) > 0:
    ax.scatter(neg_coords[:,0], neg_coords[:,1], neg_coords[:,2], 
               c='blue', s=50, alpha=0.7, label=f'-1 Antimatter ({len(neg_coords)})')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Manifested Particles: Polarity Follows Divergence', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Evaporation: Matter Returning to Void

When flux density drops below KB, manifested particles **evaporate** back to void.

In [ ]:
# Create particles and let them evaporate
CONSTANTS.KB = 1.0
CONSTANTS.DECAY_RATE = 0.05  # Fast decay

universe = Universe(size=32)
center = universe.size // 2

# Manually create some particles
particle_positions = [
    (center-3, center, center),
    (center, center-3, center),
    (center+3, center, center),
    (center, center+3, center),
    (center, center, center-3),
    (center, center, center+3),
]

for i, pos in enumerate(particle_positions):
    universe.states[pos] = 1 if i % 2 == 0 else -1
    # Give them initial flux support
    universe.flux[pos[0], pos[1], pos[2], :] = 2.0

forces.calculate_density(universe)
print(f"Created {len(particle_positions)} particles with initial flux support.")

In [ ]:
# Track evaporation
evap_history = {'t': [], 'particles': [], 'avg_density': []}

for t in range(100):
    n_particles = np.count_nonzero(universe.states != 0)
    avg_density = universe.density[universe.states != 0].mean() if n_particles > 0 else 0
    
    evap_history['t'].append(t)
    evap_history['particles'].append(n_particles)
    evap_history['avg_density'].append(avg_density)
    
    master_equation.tick(universe)

# Plot evaporation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(evap_history['t'], evap_history['particles'], 'r-', linewidth=2)
axes[0].set_xlabel('Time (ticks)', fontsize=12)
axes[0].set_ylabel('Number of Particles', fontsize=12)
axes[0].set_title('Particle Count: Evaporation', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(evap_history['t'], evap_history['avg_density'], 'g-', linewidth=2)
axes[1].axhline(CONSTANTS.KB, color='r', linestyle='--', label=f'KB = {CONSTANTS.KB}')
axes[1].set_xlabel('Time (ticks)', fontsize=12)
axes[1].set_ylabel('Avg Density at Particle Locations', fontsize=12)
axes[1].set_title('Density Decay Below Threshold', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Evaporation: Matter → Void', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Pair Production: Matter-Antimatter Creation

High-energy flux naturally produces **equal amounts** of matter and antimatter.

In [ ]:
# Run multiple pair production trials
CONSTANTS.KB = 1.0
CONSTANTS.DECAY_RATE = 0.001

n_trials = 10
positive_counts = []
negative_counts = []

for trial in range(n_trials):
    universe = Universe(size=32)
    center = universe.size // 2
    
    # Inject random high-flux region
    universe.flux[center-3:center+3, center-3:center+3, center-3:center+3, :] = np.random.randn(6, 6, 6, 3) * 3
    
    for t in range(30):
        master_equation.tick(universe)
    
    positive_counts.append(np.count_nonzero(universe.states == 1))
    negative_counts.append(np.count_nonzero(universe.states == -1))

print("Pair Production Statistics:")
print(f"  Positive (+1): {np.mean(positive_counts):.1f} ± {np.std(positive_counts):.1f}")
print(f"  Negative (-1): {np.mean(negative_counts):.1f} ± {np.std(negative_counts):.1f}")
print(f"  Ratio +/-: {np.mean(positive_counts)/np.mean(negative_counts):.2f}")

In [ ]:
# Visualize pair production statistics
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(n_trials)
width = 0.35

ax.bar(x - width/2, positive_counts, width, label='+1 (Matter)', color='red', alpha=0.7)
ax.bar(x + width/2, negative_counts, width, label='-1 (Antimatter)', color='blue', alpha=0.7)

ax.set_xlabel('Trial', fontsize=12)
ax.set_ylabel('Particle Count', fontsize=12)
ax.set_title('Pair Production: Matter-Antimatter Balance', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. The Connection to Quantum Mechanics

The manifestation process connects to the **Born rule**:

$$P(v) = \frac{|\psi(v)|^2}{||\psi||^2}$$

In FTD:
- ψ corresponds to complexified flux: ψ = Jx + iJy
- |ψ|² corresponds to flux density |J|²
- Manifestation probability depends on |J|²

This gives rise to quantum-like measurement statistics.

In [ ]:
# Demonstrate Born-rule-like statistics
# Create a flux distribution and measure where particles manifest

n_experiments = 100
manifest_positions = []

for _ in range(n_experiments):
    universe = Universe(size=32)
    center = universe.size // 2
    
    # Create a Gaussian flux distribution
    for x in range(universe.size):
        for y in range(universe.size):
            for z in range(universe.size):
                dx, dy, dz = x - center, y - center, z - center
                r = np.sqrt(dx**2 + dy**2 + dz**2)
                magnitude = 3.0 * np.exp(-r**2 / 20)
                universe.flux[x, y, z, :] = magnitude
    
    CONSTANTS.KB = 1.5
    forces.calculate_density(universe)
    
    # Run one tick to trigger manifestation
    master_equation.tick(universe)
    
    # Record manifested positions
    coords = np.argwhere(universe.states != 0)
    for coord in coords:
        manifest_positions.append(coord)

manifest_positions = np.array(manifest_positions)
print(f"Collected {len(manifest_positions)} manifestation events")

In [ ]:
# Compare manifestation distribution to flux density
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(manifest_positions) > 0:
    # Calculate radial distribution of manifestation events
    radii = np.sqrt((manifest_positions[:,0] - center)**2 + 
                    (manifest_positions[:,1] - center)**2 + 
                    (manifest_positions[:,2] - center)**2)
    
    axes[0].hist(radii, bins=20, density=True, alpha=0.7, color='blue', label='Manifestation events')
    
    # Theoretical distribution (proportional to |J|² * 4πr²)
    r_theory = np.linspace(0, 15, 100)
    # |J|² ~ exp(-r²/10) so |J|² * r² ~ r² * exp(-r²/10)
    prob_theory = r_theory**2 * np.exp(-r_theory**2 / 10)
    prob_theory /= prob_theory.sum() * (r_theory[1] - r_theory[0])
    
    axes[0].plot(r_theory, prob_theory * 5, 'r-', linewidth=2, label='Theory: |J|² × r²')
    axes[0].set_xlabel('Distance from center', fontsize=12)
    axes[0].set_ylabel('Probability Density', fontsize=12)
    axes[0].set_title('Manifestation Position Distribution', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# 2D histogram of manifestation positions
if len(manifest_positions) > 0:
    h, xedges, yedges = np.histogram2d(
        manifest_positions[:,0], manifest_positions[:,1], 
        bins=16, range=[[0, 32], [0, 32]]
    )
    im = axes[1].imshow(h.T, origin='lower', cmap='hot', extent=[0, 32, 0, 32])
    axes[1].set_xlabel('X', fontsize=12)
    axes[1].set_ylabel('Y', fontsize=12)
    axes[1].set_title('Manifestation Density (XY projection)', fontsize=12)
    plt.colorbar(im, ax=axes[1], label='Count')

plt.suptitle('Born-Rule-Like Statistics from Manifestation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Summary: The Manifestation Process

### Key Equations

| Process | Formula | Notes |
|---------|---------|-------|
| Genesis Probability | P = 1 - exp(-(ρ-KB)/KB) | For ρ > KB |
| Polarity Selection | sign(s) = sign(∇·J) | Divergence determines matter/antimatter |
| Evaporation | If ρ < KB then s → 0 | Below threshold → void |

### Physical Interpretation

- **KB** = electron mass = 0.511 MeV (derived)
- **Genesis** = pair production from high-energy flux
- **Evaporation** = particle decay / annihilation
- **Polarity balance** = matter-antimatter symmetry

### Connection to Quantum Mechanics

- Manifestation probability ~ |ψ|² (Born rule)
- Flux = wave function precursor
- Collapse = manifestation event

**Next**: See `03_forces_and_fields.ipynb` for how manifested particles interact.